## mean/max 分数聚合 + 阈值 q 扫描（不重训）\n\n用途：\n- 适用于 `baseline_transformer_postprocess.ipynb` 或 `contrastive_transformer.ipynb`。\n- 你需要先在原 notebook 里训练好 `model`，并准备好：\n  - `train_eval_loader`, `test_loader`（stride=1 的评估 loader）\n  - `train_data`, `test_data`（用于提供 seq_len）\n  - `test_labels`（0/1）\n  - `device`\n\n本 notebook 会：\n- 用同一个模型分别计算 `feature_reduce=mean` 与 `max` 的时间点级分数\n- 对每种分数做阈值分位数 `q` 扫描\n- 输出 Top-10（按 F1）和 Top-10（按 PA_F1）\n

In [ ]:
# ============ 输入检查：这些变量应当来自你的训练 notebook ============
import numpy as np
import pandas as pd
import torch

need = ['model', 'train_eval_loader', 'test_loader', 'train_data', 'test_data', 'test_labels', 'device']
missing = [k for k in need if k not in globals()]
assert not missing, f'缺少变量: {missing} （请在原 notebook 里先准备好）'

test_labels = np.asarray(test_labels, dtype=int)
print('test_labels:', test_labels.shape, 'anomaly=', int(test_labels.sum()))
print('device:', device)


In [ ]:
# ============ point-adjust + q 列表 ============
from sklearn.metrics import precision_recall_fscore_support

def point_adjust(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int).copy()
    n = len(y_true)
    i = 0
    while i < n:
        if y_true[i] == 1:
            j = i
            while j < n and y_true[j] == 1:
                j += 1
            if y_pred[i:j].max() == 1:
                y_pred[i:j] = 1
            i = j
        else:
            i += 1
    return y_pred

q_list = np.unique(np.concatenate([
    np.linspace(0.90, 0.97, 15),
    np.linspace(0.971, 0.995, 13),
    np.linspace(0.996, 0.9995, 8),
]).round(6))
print('q candidates:', len(q_list))


In [ ]:
# ============ 分数计算：feature mean vs max ============
@torch.no_grad()
def compute_point_scores_mode(model, loader, seq_len: int, feature_reduce: str = 'mean', use_amp: bool = True):
    model.eval()
    score_sum = np.zeros(int(seq_len), dtype=np.float64)
    count = np.zeros(int(seq_len), dtype=np.int64)

    amp_on = bool(use_amp and device.type == 'cuda')
    for x, idx in loader:
        x = x.to(device, non_blocking=True)
        idx = idx.numpy()
        with torch.cuda.amp.autocast(enabled=amp_on):
            x_hat = model(x)
            se = (x_hat - x).pow(2)
            if feature_reduce == 'mean':
                err = se.mean(dim=2)
            elif feature_reduce == 'max':
                err = se.max(dim=2).values
            else:
                raise ValueError("feature_reduce must be 'mean' or 'max'")

        err = err.detach().cpu().numpy()
        B, L = err.shape
        for b in range(B):
            s = int(idx[b])
            e = s + L
            score_sum[s:e] += err[b]
            count[s:e] += 1

    return score_sum / np.maximum(count, 1)


In [ ]:
# ============ 阈值 q 扫描（输出点级 + 事件级） ============
def scan_q(train_scores: np.ndarray, test_scores: np.ndarray, y_true: np.ndarray, score_name: str):
    rows = []
    for q in q_list:
        th = float(np.quantile(train_scores, float(q)))
        pred = (test_scores > th).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
        pa_pred = point_adjust(y_true, pred)
        pa_p, pa_r, pa_f1, _ = precision_recall_fscore_support(y_true, pa_pred, average='binary', zero_division=0)
        rows.append({
            'score': score_name,
            'q': float(q),
            'th': th,
            'P': float(p), 'R': float(r), 'F1': float(f1),
            'PA_P': float(pa_p), 'PA_R': float(pa_r), 'PA_F1': float(pa_f1),
        })
    return pd.DataFrame(rows)

train_len = len(train_data)
test_len = len(test_data)

train_mean = compute_point_scores_mode(model, train_eval_loader, seq_len=train_len, feature_reduce='mean')
test_mean  = compute_point_scores_mode(model, test_loader,      seq_len=test_len,  feature_reduce='mean')
df_mean = scan_q(train_mean, test_mean, test_labels, 'mean')

train_max = compute_point_scores_mode(model, train_eval_loader, seq_len=train_len, feature_reduce='max')
test_max  = compute_point_scores_mode(model, test_loader,      seq_len=test_len,  feature_reduce='max')
df_max = scan_q(train_max, test_max, test_labels, 'max')

out = pd.concat([df_mean, df_max], ignore_index=True)

print('Top-10 by point-wise F1')
display(out.sort_values(['F1', 'PA_F1'], ascending=[False, False]).head(10))

print('Top-10 by event-level PA_F1')
display(out.sort_values(['PA_F1', 'F1'], ascending=[False, False]).head(10))
